In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'

print('Racine projet :', ROOT)

df = pd.read_parquet(DATA_RAW / 'upgrade0.parquet')
df_feat = df.copy()

label_map = {
    '1A': 1, '1B': 1,
    '2A': 2, '2B': 2,
    '3A': 3, '3B': 3, '3C': 3,
    '4A': 4, '4B': 4, '4C': 4,
    '5A': 5, '5B': 5,
    '6A': 6, '6B': 6,
    '7A': 7, '7AK': 7, '7B': 7,
    '8A': 8, '8AK': 8, '8B': 8
}

climate_col = 'in.ashraeieccclimatezone2004'

if climate_col in df_feat.columns:
    df_feat[climate_col] = (
        df_feat[climate_col]
        .astype(str)
        .map(label_map)
    )
# Suppression colonnes inutiles

drop_cols = [
    c for c in [
        'in.state',
        'in.county',
        'in.city',
        'in.weather_file_city'
    ]
    if c in df_feat.columns
]

df_feat.drop(columns=drop_cols, inplace=True)


# Colonnes catégorielles

one_hot_cols = [
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.heating_fuel',
    'in.geometry_building_type_recs',
    'in.census_region'
]

one_hot_cols = [c for c in one_hot_cols if c in df_feat.columns]

# Conversion category = moins de RAM
for c in one_hot_cols:
    df_feat[c] = df_feat[c].astype('category')


# One-hot encoding

df_feat = pd.get_dummies(
    df_feat,
    columns=one_hot_cols,
    dummy_na=False,
    drop_first=False
)


# Gestion des NaN

nan_pct = df_feat.isna().mean()

# Colonnes avec >30% NaN
drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()

df_feat.drop(columns=drop_nan_cols, inplace=True)

# Colonnes numériques
num_cols = df_feat.select_dtypes(include=[np.number]).columns

# Médianes
medians = df_feat[num_cols].median()

# Colonnes à remplir
fill_cols = nan_pct[
    (nan_pct > 0) & (nan_pct < 0.05)
].index.intersection(num_cols)

for c in fill_cols:
    df_feat[c] = df_feat[c].fillna(medians[c])

# Suppression lignes restantes avec NaN
df_feat.dropna(inplace=True)



# Sauvegarde

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

df_feat.to_parquet(
    DATA_PROCESSED / 'metadata_features.parquet',
    index=False
)

print("Dataset initial :", df.shape)
print("Dataset final :", df_feat.shape)

# Taille mémoire
mem_gb = df_feat.memory_usage(deep=True).sum() / 1e9
print(f"Mémoire utilisée : {mem_gb:.2f} GB")

Racine projet : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax
Dataset initial : (549971, 771)
Dataset final : (483334, 768)
Mémoire utilisée : 3.59 GB


convertir les colonnes HourX :in.range_spot_vent_hour ,,,,in.bathroom_spot_vent_hour

In [2]:
import re
def exact_hour_to_int(hour_str):
    if pd.isna(hour_str):
        return np.nan
    h=str(hour_str)
    # Cherche "Hour0", "Hour12", etc.
    m = re.search(r'Hour(\d+)', h)
    if m:
        return int(m.group(1))
    return np.nan
'''
cols_to_encode_hour = [ 'in.bathroom_spot_vent_hour','in.range_spot_vent_hour']
for col in cols_to_encode_hour:
    df_feat[col] = df_feat[col].apply(exact_hour_to_int)

print(df_feat[cols_to_encode_hour].head())
print(df_feat[cols_to_encode_hour].dtypes)
'''    

"\ncols_to_encode_hour = [ 'in.bathroom_spot_vent_hour','in.range_spot_vent_hour']\nfor col in cols_to_encode_hour:\n    df_feat[col] = df_feat[col].apply(exact_hour_to_int)\n\nprint(df_feat[cols_to_encode_hour].head())\nprint(df_feat[cols_to_encode_hour].dtypes)\n"

convertir des porcentage 120% ==>1.2 : in.clothes_washer_usage_level,,,,,in.dishwasher_usage_level,,,in.clothes_dryer_usage_level,,,,,in.cooking_range_usage_level,,,,in.hot_water_fixtures,,,,in.lighting_interior_use,,,,,in.lighting_other_use,,,,,in.refrigerator_usage_level

In [1]:
import re
import numpy as np
import pandas as pd
def pourcentage(pr):
    if pd.isna(pr):
        return np.nan
    p=re.search(r'(\d+)', str(pr))
    return int(p.group(1))/100 if p else np.nan
'''
cols_to_encode_pourcentage = [ 'in.clothes_dryer_usage_level','in.clothes_washer_usage_level',
                              'in.dishwasher_usage_level','in.cooking_range_usage_level',
                              'in.hot_water_fixtures','in.lighting_interior_use','in.lighting_other_use',
                              'in.refrigerator_usage_level'
                              ]
for col in cols_to_encode_pourcentage:
    df_feat[col] = df_feat[col].apply(pourcentage)

print(df_feat[cols_to_encode_pourcentage].head())
print(df_feat[cols_to_encode_pourcentage].dtypes)
'''

"\ncols_to_encode_pourcentage = [ 'in.clothes_dryer_usage_level','in.clothes_washer_usage_level',\n                              'in.dishwasher_usage_level','in.cooking_range_usage_level',\n                              'in.hot_water_fixtures','in.lighting_interior_use','in.lighting_other_use',\n                              'in.refrigerator_usage_level'\n                              ]\nfor col in cols_to_encode_pourcentage:\n    df_feat[col] = df_feat[col].apply(pourcentage)\n\nprint(df_feat[cols_to_encode_pourcentage].head())\nprint(df_feat[cols_to_encode_pourcentage].dtypes)\n"

Convertir les colonnes de temperature (60F->60) : in.cooling_setpoint,,,,,in.cooling_setpoint_offset_magnitude,,,in.heating_setpoint,,,,in.heating_setpoint_offset_magnitude
NB:il faut traiter le NAN après

In [3]:
import re
import numpy as np
import pandas as pd

def temperature(temp_str):
    if pd.isna(temp_str):
        return np.nan
    
    s = str(temp_str)

    t = re.search(r'(\d+)', s)
    
    if not t:
        return np.nan

    temp = float(t.group(1))

    # Conversion Fahrenheit -> Celsius
    if 'f' in s.lower():
        return (temp - 32) * 5.0 / 9.0
    else:
        return temp
''' 
cols_to_encode_temperature = [ 'in.cooling_setpoint','in.cooling_setpoint_offset_magnitude','in.heating_setpoint','in.heating_setpoint_offset_magnitude']
for col in cols_to_encode_temperature:
    df_feat[col] = df_feat[col].apply(temperature)      
print(df_feat[cols_to_encode_temperature].head())
print(df_feat[cols_to_encode_temperature].dtypes)  
'''  

" \ncols_to_encode_temperature = [ 'in.cooling_setpoint','in.cooling_setpoint_offset_magnitude','in.heating_setpoint','in.heating_setpoint_offset_magnitude']\nfor col in cols_to_encode_temperature:\n    df_feat[col] = df_feat[col].apply(temperature)      \nprint(df_feat[cols_to_encode_temperature].head())\nprint(df_feat[cols_to_encode_temperature].dtypes)  \n"

In [ ]:
in.cooling_setpoint_offset_period,,,,,in.heating_setpoint_offset_period

In [5]:

def offset_period(val):
    """
    Retourne un dict avec 3 features numériques extraites de la valeur string.
    Exemples :
        "Day Setup +2h"                    → period=1, direction=1,  hours=2
        "Night Setback -3h"                → period=2, direction=-1, hours=3
        "Day Setup and Night Setback +1h"  → period=3, direction=0,  hours=1
        "Day and Night Setup -4h"          → period=3, direction=1,  hours=4
        "None"                             → period=0, direction=0,  hours=0
    """
    if pd.isna(val) or str(val).strip().lower() == 'none':
        return {'offset_period_type': 0,
                'offset_direction':   0,
                'offset_hours':       0.0}

    s = str(val).strip()

    # Type de période : Day=1, Night=2, Day+Night=3
    has_day   = 'day'   in s.lower()
    has_night = 'night' in s.lower()
    if has_day and has_night:
        period_type = 3
    elif has_day:
        period_type = 1
    elif has_night:
        period_type = 2
    else:
        period_type = 0

    # Direction : Setup=+1 , Setback=-1
    has_setup   = 'setup'   in s.lower()
    has_setback = 'setback' in s.lower()
    if has_setup and has_setback:
        direction = 0   
    elif has_setup:
        direction = 1
    elif has_setback:
        direction = -1
    else:
        direction = 0
#+3h ->3.0
    m = re.search(r'([+-]?\d+)h', s)
    hours = abs(float(m.group(1))) if m else 0.0

    return {
        'offset_period_type': period_type,
        'offset_direction':   direction,
        'offset_hours':       hours,
    }
'''
OFFSET_COL = ['in.cooling_setpoint_offset_period','in.heating_setpoint_offset_period']

for col in OFFSET_COL:

    parsed = df_feat[col].apply(offset_period)
    parsed_df = pd.DataFrame(parsed.tolist(), index=df_feat.index)

    df_feat[f'{col}_type'] = parsed_df['offset_period_type'].astype('int8')
    df_feat[f'{col}_direction']   = parsed_df['offset_direction'].astype('int8')
    df_feat[f'{col}_hours']       = parsed_df['offset_hours'].astype('float32')

    # Supprimer la colonne string originale
    df_feat.drop(columns=[col], inplace=True)

    print('Colonnes créées :')
    print(df_feat[[f'{col}_type',
                   f'{col}_direction',
                   f'{col}_hours']].value_counts().head(10))


# Traitement NaN 
# Ces 3 colonnes n'ont pas de NaN (None → 0 géré dans la fonction)
# Vérification

new_cols = [f'{col}_type' for col in OFFSET_COL] + \
           [f'{col}_direction' for col in OFFSET_COL] + \
           [f'{col}_hours' for col in OFFSET_COL]

print('\nNaN résiduels :')
print(df_feat[new_cols].isna().sum())

df_feat[new_cols] = df_feat[new_cols].fillna(0)


print(f'\n{"Colonne":<35} {"Type":>10} {"Uniques":>10} {"% NaN":>8}')
print('-' * 70)
for col in new_cols:
    print(f'{col:<35} '
          f'{str(df_feat[col].dtype):>10} '
          f'{df_feat[col].nunique():>10} '
          f'{df_feat[col].isna().mean()*100:>7.1f}%')

'''

'\nOFFSET_COL = [\'in.cooling_setpoint_offset_period\',\'in.heating_setpoint_offset_period\']\n\nfor col in OFFSET_COL:\n\n    parsed = df_feat[col].apply(offset_period)\n    parsed_df = pd.DataFrame(parsed.tolist(), index=df_feat.index)\n\n    df_feat[f\'{col}_type\'] = parsed_df[\'offset_period_type\'].astype(\'int8\')\n    df_feat[f\'{col}_direction\']   = parsed_df[\'offset_direction\'].astype(\'int8\')\n    df_feat[f\'{col}_hours\']       = parsed_df[\'offset_hours\'].astype(\'float32\')\n\n    # Supprimer la colonne string originale\n    df_feat.drop(columns=[col], inplace=True)\n\n    print(\'Colonnes créées :\')\n    print(df_feat[[f\'{col}_type\',\n                   f\'{col}_direction\',\n                   f\'{col}_hours\']].value_counts().head(10))\n\n\n# Traitement NaN \n# Ces 3 colonnes n\'ont pas de NaN (None → 0 géré dans la fonction)\n# Vérification\n\nnew_cols = [f\'{col}_type\' for col in OFFSET_COL] +            [f\'{col}_direction\' for col in OFFSET_COL] +        

convesion des colonnes : 'in.cooling_unavailable_days' 
"Never"      → 0
"1 day"      → 1
"3 days"     → 3
"1 week"     → 7
"2 weeks"    → 14
"1 month"    → 30
"3 months"   → 90
"Year round" → 365  

In [ ]:
def unavailable_days(days_str):
    if pd.isna(days_str):
        return np.nan
    s = str(days_str).strip().lower()
    if s == 'never':       return 0
    if s == 'year round':  return 365
    m_month = re.search(r'(\d+)\s*month', s)
    if m_month: return int(m_month.group(1)) * 30
    m_week  = re.search(r'(\d+)\s*week',  s)
    if m_week:  return int(m_week.group(1)) * 7
    m_day   = re.search(r'(\d+)\s*day',   s)
    if m_day:   return int(m_day.group(1))
    return np.nan

for col in ['in.cooling_unavailable_days', 'in.heating_unavailable_days']:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(unavailable_days).astype('float32')
        # NaN → médiane (cas rares)
        median_val = df_feat[col].median()
        df_feat[col] = df_feat[col].fillna(median_val)
        print(f'{col} → float32 ✓  (médiane={median_val:.0f} jours)')


pour convertir la colonnre 'in.corridor' mais aussi pour toutes les colonnes de type boolean 

In [ ]:
def corridor_to_int(corridor_str):
    if pd.isna(corridor_str):
        return np.nan
    s = str(corridor_str).lower()
    if s == 'Double-Loaded Interior':   return 0
    if s == 'Not Applicable': return 1
    return np.nan

df_feat['in.corridor']=df_feat['in.corridor'].apply(corridor_to_int)




conversion de 'in.duct_leakage_and_insulation' ,on extrait 2 feature le leakage (%) et l-isolation (R-value -> int)

In [ ]:
def duct_leakage_pourcentage():
    if pd.isna(val) or val == "None":
        return np.nan

    m = re.search(r'(\d+)%', str(val))
    return float(m.group(1)) / 100 if m else np.nan


def parse_duct_r(val):
    if pd.isna(val) or val == "None":
        return 0  

    s = str(val)

    if "R-4" in s:
        return 4
    if "R-6" in s:
        return 6
    if "R-8" in s:
        return 8
    if "uninsulated" in s.lower():
        return 0

    return np.nan



